[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/origami/origami.ipynb)

# Folding a Kabuto Helmet

This example folds a flat square of paper into a traditional samurai helmet, a kabuto, through a sequence of straight creases and rigid turns. Each stage defines a crease plane by bringing one corner onto another or one folded edge onto another, splits the crossed polygons into moving and stationary pieces, and turns the selected layers about their common hinge line while carrying topological layer heights so stacked flaps never pass through stationary paper.

In projective geometric algebra, points are modelled as antivectors, joined lines as antibivectors, crease planes as vectors, and hinge axes as bivectors. The crease plane that folds one unit-weight point onto another is their perpendicular bisector `((first & second) | (first + second)).normalized()`, and the crease plane that folds one oriented line onto another through their intersection is `((first + second) | intersection).normalized()`. Meeting a crease plane with the sheet plane gives the hinge line `crease ^ sheet`, whose exponential `(hinge * (angle / 2)).exp()` is the motor turning the moving faces, while any polygon edge crossing the crease splits at `(tail & head) ^ crease`. The construction follows the [origami example](https://github.com/enkimute/ganja.js/blob/master/examples/example_pga3d_origami.html) of ganja.js by Steven De Keninck.

In Gibbs–Heaviside vector notation, the perpendicular bisector of $\mathbf a$ and $\mathbf b$ is $(\mathbf b - \mathbf a)\cdot\mathbf x = \tfrac12(|\mathbf b|^2 - |\mathbf a|^2)$, clipping an edge from $\mathbf p_0$ to $\mathbf p_1$ evaluates $t = (\mathbf n\cdot\mathbf p_0 - d) / (\mathbf n\cdot(\mathbf p_0 - \mathbf p_1))$, and folding by $\theta$ around a hinge through $\mathbf p$ along unit direction $\mathbf u = (\mathbf n \times \mathbf n_s)/|\mathbf n \times \mathbf n_s|$ maps $\mathbf x$ to $\mathbf p + ((\mathbf x - \mathbf p)\cos\theta + (\mathbf u \times (\mathbf x - \mathbf p))\sin\theta + \mathbf u(\mathbf u\cdot(\mathbf x - \mathbf p))(1 - \cos\theta))$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack, where
from numga.algebras import PGA3D
from numga.sparse import Sparse
from numga.topology import Topology
from examples.geometry.origami import render, scenarios
from examples.util.animation import save_figure, show_states

ga = PGA3D
mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                      # Antivector-
Direction = Point.derive.degenerate                 # Antivector-
Line = ga.gatype.antibivector()                     # Antibivector-
Hinge = ga.gatype.bivector()                        # Bivector+
Plane = ga.gatype.vector()                          # Vector+
origin = mv.point([0, 0, 0])                        # [] Point-

PaperState = tuple[Point, np.ndarray, Direction, Plane]

## Square sheet

A flat square sits in the horizontal plane with its four corners generated in cyclic order from the upper-left corner by quarter-turn motors, and carries an initial zero layer offset in the sheet's normal direction.

In [ ]:
side = scenarios.side
sheet = mv.z                                                                    # [] Plane+

In [ ]:
corner = mv.point([-side, side, 0])                                             # [] Point-
quarter_turns = (mv.xy * (-np.arange(4) * np.pi / 4)).exp()                     # [4] Motor+
corners = quarter_turns >> corner                                               # [4] Point-
upper_left, lower_left, lower_right, upper_right = corners                      # [] Point-, [] Point-, [] Point-, [] Point-
offsets = mv.direction([[0, 0, 0]]).field()                                     # Direction-[1]
square_state = (corners.field(), np.array([len(corners)]), offsets, sheet)
save_figure(render.paper_figure(square_state), "origami_square");

The orange front side of the unfolded square faces upward before any creases split its boundary.

## Crease bisectors and polygon splitting

Bringing one corner onto another folds along their perpendicular bisector plane, while bringing two intersecting lines together folds along their angle bisector plane. Splitting the selected polygons along a crease inserts intersection vertices on crossed boundary edges and duplicates crease vertices so the moving and stationary halves turn independently while carrying integer layer offsets past the stationary stack.

In [ ]:
crease_tolerance = scenarios.crease_tolerance
first_faces = (0,)
preview_angle = np.pi / 2

In [ ]:
def point_bisector(first: Point, second: Point) -> Plane:
    """The perpendicular bisector plane of two unit-weight points folds one onto the other (`[...] Plane+`)."""
    return ((first & second) | (first + second)).normalized()                   # [...] Plane+


def line_bisector(first: Line, second: Line, intersection: Point) -> Plane:
    """The angle bisector plane of two oriented lines crossing at `intersection` folds one onto the other (`[...] Plane+`)."""
    first, second = first.normalized(), second.normalized()                     # [...] Line-, [...] Line-
    return ((first + second) | intersection).normalized()                       # [...] Plane+

In [ ]:
def polygon_faces(counts: np.ndarray) -> tuple[np.ndarray, ...]:
    """Contiguous corner counts partition the flat corner sequence into one index array per face."""
    return tuple(np.split(np.arange(int(counts.sum())), np.cumsum(counts)[:-1]))


def split_polygons(points: Point, counts: np.ndarray, offsets: Direction, crease: Plane,
                   selected: tuple[int, ...], tolerance: float) -> tuple[Point, np.ndarray, Direction, np.ndarray]:
    """Cutting `selected` faces along `crease` inserts edge intersections and duplicates crease corners into moving then stationary polygons."""
    faces = polygon_faces(counts)
    F = len(counts)
    loops = Topology.loops(counts)
    V, E = loops.n_elements
    # Each polygon's closed boundary loop pairs every corner with the next corner around its face.
    tails, heads = loops.incidence[1, 0] * points                               # Point-[E], Point-[E]
    distances = crease & stack([tails, heads])                                  # [2] Scalar+[E]
    negative, positive = distances < -tolerance, distances > tolerance
    crossing = (negative[0] & positive[1]) | (positive[0] & negative[1])

    # Only crossing edges divide by their intersection weight, so edges parallel to the crease stay finite.
    cuts = (tails & heads) ^ crease                                             # Point-[E]
    unit_weights = mv.scalar(np.ones((V, 1))).field()                           # Scalar+[V]
    cut_points = where(crossing, cuts / where(crossing, mv.w & cuts, unit_weights), points)  # Point-[V]
    corners_and_cuts = stack([points, cut_points])                              # [2] Point-[V]
    vertex_mask = np.stack([np.ones_like(crossing), crossing], axis=-1)
    pair_distances = crease & corners_and_cuts                                  # [2] Scalar+[V]
    pair_negative, pair_positive = (pair_distances < -tolerance).T, (pair_distances > tolerance).T
    corners_only = np.array([True, False])

    # Duplicating crease vertices lets moving and stationary pieces turn independently about their shared edge.
    piece_masks, turning, parents = [], [], []
    for index, face in enumerate(faces):
        if index in selected:
            piece_masks.append(vertex_mask[face] & ~pair_positive[face])
            turning.append(True)
            parents.append(index)
            piece_masks.append(vertex_mask[face] & ~pair_negative[face])
            turning.append(False)
            parents.append(index)
        else:
            piece_masks.append(vertex_mask[face] & corners_only)
            turning.append(False)
            parents.append(index)

    parent_indices = np.array(parents)
    corner, at_cut = np.nonzero(np.concatenate(piece_masks))
    corner_indices = np.concatenate([faces[parent] for parent in parent_indices])[corner]
    split_counts = np.array([mask.sum() for mask in piece_masks])
    picked_pairs = Sparse.selection(corner_indices, V) * corners_and_cuts       # [2] Point-[V_split]
    split_points = where(at_cut == 1, picked_pairs[1], picked_pairs[0])         # Point-[V_split]
    split_offsets = Sparse.selection(parent_indices, F) * offsets               # Direction-[F_split]
    return split_points, split_counts, split_offsets, np.array(turning)

In [ ]:
def folded_state(points: Point, counts: np.ndarray, offsets: Direction, sheet: Plane,
                 hinge: Hinge, turning: np.ndarray, angle: float) -> PaperState:
    """Turning the `turning` faces by `angle` about `hinge` transports their corners and carries their layer offsets past the stationary stack."""
    hinge = hinge.normalized()                                                  # [] Hinge+
    F = len(counts)
    face_motors = (hinge * (angle / 2 * turning)).exp().field()                 # Motor+[F]
    corner_motors = Sparse.selection(np.repeat(np.arange(F), counts), F) * face_motors  # Motor+[V]
    turned_points = corner_motors >> points                                     # Point-[V]

    # Signed layer heights along the sheet normal place the offset pivot halfway above the two stacks' top layers.
    normal_direction = (sheet | origin) ^ mv.w                                  # [] Direction-
    normal = normal_direction * (sheet & normal_direction)                      # [] Direction-
    heights = (sheet & offsets) / (sheet & normal) * np.sign(angle)             # Scalar+[F]
    on_turning = Sparse.selection(np.flatnonzero(turning), F)                   # Scalar+[F_turning <- F]
    on_stationary = Sparse.selection(np.flatnonzero(~turning), F)               # Scalar+[F_stationary <- F]
    pivot_height = ((on_turning * heights).max() + (on_stationary * heights).max() + 1) / 2  # [] Scalar+
    pivot = normal * (pivot_height * np.sign(angle))                            # [] Direction-
    turned_offsets = (face_motors >> (offsets - pivot)) + pivot                 # Direction-[F]
    return turned_points, counts, turned_offsets, sheet


first_crease = point_bisector(upper_left, lower_right)                          # [] Plane+
first_hinge = first_crease ^ sheet                                              # [] Hinge+
first_points, first_counts, first_offsets, first_turning = split_polygons(
    square_state[0], square_state[1], square_state[2], first_crease, first_faces, crease_tolerance,
)
preview_state = folded_state(
    first_points, first_counts, first_offsets, sheet, first_hinge, first_turning, preview_angle,
)
save_figure(render.paper_figure(preview_state), "origami_first_fold");

Halfway through the diagonal fold, the upper-left triangle stands at a right angle to the stationary lower-right triangle along the inked diagonal hinge, shaded dark where its orange front turns away from the camera.

## Kabuto crease recipe and folding sequence

Each selected polygon splits into its moving piece followed by its stationary piece, so face indices after each stage address individual layers within multi-layer flaps. Point bisectors fold the main triangle and brim flaps, two folds close and reopen to leave guide creases, and line bisectors swing out the two horns before the final flap tucks behind the helmet.

In [ ]:
selections = scenarios.selections
peaks = scenarios.peaks
ends = scenarios.ends
n_half_steps = scenarios.n_half_steps
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
centre = corners.mean(axis=0)                                                   # [] Point-
top = (upper_right + upper_left) / 2                                            # [] Point-
left = (upper_left + lower_left) / 2                                            # [] Point-
quarter = (upper_left + centre) / 2                                             # [] Point-
upper_centre = (centre + top) / 2                                               # [] Point-
left_centre = (centre + left) / 2                                               # [] Point-
eighth = (centre + quarter) / 2                                                 # [] Point-

# Point coincidences set the main flaps; line coincidences open the two horns.
creases = stack([                                                               # [11] Plane+
    point_bisector(upper_left, lower_right),
    point_bisector(upper_right, lower_right),
    point_bisector(lower_left, lower_right),
    point_bisector(centre, upper_left),
    point_bisector(top, centre),
    point_bisector(left, centre),
    line_bisector(left_centre & quarter, quarter & centre, quarter),
    line_bisector(upper_centre & quarter, quarter & centre, quarter),
    point_bisector(eighth, upper_left),
    point_bisector(centre, upper_left),
    point_bisector(centre, upper_left),
])

In [ ]:
def folding_states(initial_state: PaperState, creases: Plane, selections: tuple[tuple[int, ...], ...],
                   peaks: np.ndarray, ends: np.ndarray, n_half_steps: int, tolerance: float) -> Iterator[PaperState]:
    """Yields the paper state `(points, counts, offsets, sheet)` across every eased half-stage of the folding sequence."""
    points, counts, offsets, sheet = initial_state
    yield points, counts, offsets, sheet
    fractions = np.linspace(0, 1, n_half_steps + 1)[1:]
    fractions = fractions * fractions * (3 - 2 * fractions)
    for crease, selected, peak, end in zip(creases, selections, peaks, ends):
        split_points, split_counts, split_offsets, turning = split_polygons(
            points, counts, offsets, crease, selected, tolerance,
        )
        hinge = crease ^ sheet                                                  # [] Hinge+
        for angle in np.concatenate([peak * fractions, peak + (end - peak) * fractions]):
            points, counts, offsets, sheet = folded_state(
                split_points, split_counts, split_offsets, sheet, hinge, turning, angle,
            )
            yield points, counts, offsets, sheet


states = show_states(
    folding_states(square_state, creases, selections, peaks, ends, n_half_steps, crease_tolerance),
    render.folding_animation,
    name="origami_kabuto",
    duration_ms=frame_duration_ms,
);

Orange front faces and cream reverse faces alternate across the folded flaps as the two cream horns open outward above the orange brim and the back flap folds underneath.

In [ ]:
# checks
preview_points, _, _, _ = preview_state
assert ((first_crease & upper_left) + (first_crease & lower_right)).abs() < 1e-12
assert (((first_hinge.normalized() * (np.pi / 2)).exp() >> upper_left) & lower_right).antinorm() < 1e-10

final_points, _, _, final_sheet = states[-1]
horn_tips = mv.point([[-1, 1 - np.sqrt(2), 0], [np.sqrt(2) - 1, 1, 0]])         # [2] Point-
tip_distances = (final_points.batch()[:, None] & horn_tips).antinorm()          # [V, 2] Scalar+
assert (tip_distances.min(axis=0) < 1e-6).all()
assert ((final_sheet & final_points).abs() < 1e-11).all()